# P9-IMPL-WILDFIRE-INGEST — NIFC Wildfire Perimeter Ingestion Pipeline

**Author:** @data-engineer | **Date:** 2026-10-22  
**Schema:** `cdm_tmforum.provable_resilience_intelligence`  
**Source:** [NIFC WFIGS Open Data — Interagency Fire Perimeters](https://data-nifc.opendata.arcgis.com/)  
**Refresh:** 2-hour cadence (perimeters update as fires evolve)  
**SID Domain:** Common → GeographicPlace → HazardEvent (wildfire extension)  

## Pipeline Architecture
```
NIFC WFIGS ArcGIS FeatureServer → Fetch+Parse → GeoJSON POLYGON→WKT → stg_nifc_wildfire (Bronze)
                                                                      ↓ MERGE
                                                                 geographic_hazard_event (Gold)
                                                                      ↓ SPATIAL JOIN
                                                                 hazard_span_intersection
```

## Key Transforms
- ArcGIS Epoch ms → TIMESTAMP
- Acres → estimated_radius_km = sqrt(acres * 0.00404686 / π)
- GeoJSON rings → WKT POLYGON/MULTIPOLYGON
- Incident status → PRI severity mapping (Active+low containment=critical, Active+high=high, Contained=medium, Out=low)

## Lakelink Fiber — Provable Resilience Intelligence

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_nifc_wildfire (
  irwin_id              STRING        NOT NULL  COMMENT 'IRWIN (Integrated Reporting of Wildland-Fire Information) unique fire ID',
  perimeter_id          STRING        NOT NULL  COMMENT 'Unique perimeter feature ID from WFIGS. One fire may have multiple perimeters over time.',
  fire_name             STRING        NOT NULL  COMMENT 'Official incident name (e.g. Park Fire, Bridge Fire)',
  fire_type             STRING                  COMMENT 'Incident type: Wildfire | Prescribed Fire | Complex',
  fire_cause            STRING                  COMMENT 'Cause category: Human | Lightning | Unknown | Under Investigation',
  incident_status       STRING                  COMMENT 'Incident status: Active | Contained | Controlled | Out',
  containment_pct       DOUBLE                  COMMENT 'Percent contained (0-100). NULL if not reported.',
  acres_burned          DOUBLE                  COMMENT 'GIS-calculated acres within the fire perimeter',
  discovery_acres       DOUBLE                  COMMENT 'Acres at time of discovery/first report',
  perimeter_wkt         STRING        NOT NULL  COMMENT 'WKT POLYGON or MULTIPOLYGON of fire perimeter boundary. WGS84.',
  centroid_lat          DOUBLE        NOT NULL  COMMENT 'Centroid latitude of fire perimeter for proximity queries',
  centroid_lon          DOUBLE        NOT NULL  COMMENT 'Centroid longitude of fire perimeter for proximity queries',
  estimated_radius_km   DOUBLE                  COMMENT 'Derived: sqrt(acres_burned * 0.00404686 / PI) km. Circular-equivalent radius for buffer zone.',
  discovery_date        TIMESTAMP     NOT NULL  COMMENT 'Fire discovery or first-report datetime (UTC)',
  containment_date      TIMESTAMP               COMMENT 'Date fire reached full containment (UTC). NULL if ongoing.',
  perimeter_date        TIMESTAMP     NOT NULL  COMMENT 'Date/time this perimeter was captured (UTC)',
  personnel_count       INT                     COMMENT 'Number of personnel assigned to the incident',
  structures_threatened INT                     COMMENT 'Number of structures threatened',
  structures_destroyed  INT                     COMMENT 'Number of structures destroyed',
  source_url            STRING        NOT NULL  COMMENT 'NIFC WFIGS feature service URL',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original GeoJSON feature (for audit/replay)'
)
COMMENT 'Staging: NIFC active wildfire perimeters from WFIGS Open Data. One row per perimeter snapshot. Refreshed every 2 hours. Source for geographic_hazard_event wildfire enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT2H',
  'source.api' = 'data-nifc.opendata.arcgis.com/WFIGS',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
)